# PigeonPilot — Interactive Playground

Load a **finished** named run from `Models.ipynb` (Step 10), review test metrics, then draw a displacement path and watch the pigeon fly out and home.

**Contract:** the readout predicts a **home heading bin** (10°), not a full return polyline. The homebound animation is a straight ray (predicted direction × true home distance). True home = blue, predicted = magenta.

Requires: `%matplotlib widget` (`ipympl`) + at least one run under `outputs/checkpoints/`.


### 1. Imports + pick a run


In [ ]:
%matplotlib widget

from IPython.display import Markdown, display

from pigeonpilot.snn import list_runs, load_run
from pigeonpilot.playground import launch_playground

runs = list_runs()
assert runs, "No checkpoints yet — run Models.ipynb Step 10 (save_run) first."

display(Markdown("### Available runs"))
for r in runs:
    mark = " ← latest" if r["is_latest"] else ""
    display(Markdown(f"- `{r['name']}` · n_res={r['n_reservoir']}{mark}"))

# Change this to a concrete name, e.g. "n10000_main", or keep "latest"
RUN_NAME = "latest"
print("will load:", RUN_NAME)


### 2. Load checkpoint + show jury metrics


In [ ]:
bundle = load_run(RUN_NAME)
cfg = bundle.config
metrics = bundle.metrics

display(Markdown(
    f"### Loaded `{RUN_NAME}`\n"
    f"- reservoir size: **{cfg.n_reservoir}**\n"
    f"- encoding: v=`{cfg.encoding_velocity:.4f}`, dt=`{cfg.encoding_dt}`, "
    f"rate=`{cfg.input_rate_hz}` Hz, silence=`{cfg.trailing_silence}`\n"
    f"- ridge α=`{cfg.ridge_alpha}`"
))

summary = metrics.get("summary") or {}
if summary:
    lines = ["### Test angular error (from training run)\n"]
    for name in ("A", "B"):
        if name not in summary:
            continue
        s = summary[name]
        lines.append(
            f"- **Pigeon {name}**: mean error {s['mean_deg']:.1f}° ± {s['std_deg']:.1f}° "
            f"| exact-bin acc {100 * s['exact_acc']:.1f}%"
        )
    by_diff = metrics.get("by_difficulty") or {}
    if by_diff:
        lines.append("\n| difficulty | A mean ° | B mean ° | n |")
        lines.append("|---|---:|---:|---:|")
        for diff, row in by_diff.items():
            lines.append(
                f"| {diff} | {row['A_mean_deg']:.1f} | {row['B_mean_deg']:.1f} | {row['n']} |"
            )
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("_No metrics stored in this checkpoint — demo still works._"))


### 3. Draw → Fly → infer home

1. Draw with the mouse starting near the green home star
2. Click **Fly** — outbound animation, then reservoir inference, then predicted home ray
3. **Clear** to try another path

Set `model="both"` for A vs B side-by-side (slower: two reservoir runs).


In [ ]:
# model: "A" | "B" | "both"
playground = launch_playground(bundle, model="A", field_half=8.0)
playground.fig  # show figure in widget backend
